In [ ]:
%sql
/* 
(not sure what will happen here on execute with no actual sql to run)
noting that this ipynb approach is NOT necessary or even recommended - an organic production instance would probably use 
a collection of SQL files instead - I just like this for demo simplicity
pipeline overview:
- mock data already instantiated upstream (raw schemas as source system)
- streaming tables (create or refresh) touching mock data - bronze
- streaming tables (create or refresh) touching bronze data - silver
- mv joining silver tables together - silver
- mv aggregating joined silver data - gold
*/

In [ ]:
%sql
CREATE OR REFRESH STREAMING TABLE main.bronze.items AS
    SELECT * FROM STREAM main.raw.items
;

CREATE OR REFRESH STREAMING TABLE main.bronze.transactions AS
    SELECT * FROM STREAM main.raw.transactions
;

In [ ]:
%sql
-- pretend we have some sanitation sql here
CREATE OR REFRESH STREAMING TABLE main.silver.dim_items AS
    SELECT * FROM STREAM main.bronze.items
;

CREATE OR REFRESH STREAMING TABLE main.silver.fct_transactions AS
    SELECT * FROM STREAM main.bronze.transactions
;

In [ ]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW main.silver.v_transactions AS
    SELECT 
        t.txn_id
        , t.item_id
        , i.name as item_name
        , i.category as item_category
        , i.price as unit_price
        , t.quantity
        , (t.quantity * i.price) as line_price
        , t.txn_time
    FROM
        main.silver.fct_transactions t
        INNER JOIN main.silver.dim_items i ON t.item_id = i.id
;

In [ ]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW main.gold.v_transactions AS
    SELECT * FROM main.silver.v_transactions        
;

CREATE OR REPLACE MATERIALIZED VIEW main.gold.v_daily_totals AS
    SELECT
        t.txn_time::DATE as date
        , COUNT(*) as count_txns
        , SUM(t.quantity) as total_quantity
        , SUM(t.line_price) as total_price
    FROM 
        main.silver.v_transactions
;